<a href="https://colab.research.google.com/github/mayayonathan/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-25%20%E2%80%94%20Cleaning%20Gauntlet%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
before_rows = len(df)
removed = df.duplicated().sum()
df = df.drop_duplicates().copy()

log('duplicates', 'dropped exact duplicate rows', removed)

[duplicates] dropped exact duplicate rows (15 row(s))


### TODO 2 — clean `price` -> float

In [4]:
df['price'] = (
    df['price'].astype(str).str.strip().str.replace('$', '', regex=False).astype(float)
)

assert df['price'].dtype == float
log('price to numeric', "stripped '$' and stray whitespace, converted to float", len(df))

[price to numeric] stripped '$' and stray whitespace, converted to float (300 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')

missing = df['qty'].isna().sum()
negative = (df['qty'] < 0).sum()

# Decision: a missing qty means we don't know what was sold, so drop it
# rather than guess a number.
df = df[df['qty'].notna()]
log('missing quantity', 'dropped rows with no quantity (unknown units sold)', missing)

# Decision: a negative qty is a refund, not a new sale. Drop it rather than
# netting it against sales, so revenue reflects units actually delivered.
df = df[df['qty'] >= 0]
log('negative quantity', 'dropped refund rows (negative units) rather than netting them', negative)

[missing quantity] dropped rows with no quantity (unknown units sold) (12 row(s))
[negative quantity] dropped refund rows (negative units) rather than netting them (13 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
print(df['item'].value_counts())

ITEM_MAP = {
    'cheeseburger': 'cheeseburger',
    'cheese burger': 'cheeseburger',
    'foam finger': 'foam finger',
    'rain poncho': 'rain poncho',
}
before_n = df['item'].nunique()
df['item'] = df['item'].str.strip().str.lower().map(ITEM_MAP)
after_n = df['item'].nunique()

print(df['item'].value_counts())

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
item
foam finger     97
rain poncho     91
cheeseburger    87
Name: count, dtype: int64


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
print(df['category'].value_counts())

# Decision: Apparel is folded into Merch. It is not a spelling variant of Merch,
# it is a business call that "Merch" covers all non-food, non-raingear goods.
CATEGORY_MAP = {
    'food': 'Food',
    'merch': 'Merch',
    'apparel': 'Merch',
    'raingear': 'RainGear',
}
before_n = df['category'].nunique()
df['category'] = (
    df['category'].str.lower().str.strip().str.replace(r'[^a-z]', '', regex=True).map(CATEGORY_MAP)
)
after_n = df['category'].nunique()

print(df['category'].value_counts())
log('category cleanup', f'normalized case/punctuation, folded Apparel into Merch ({before_n} -> {after_n} categories)', len(df))

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
category
Food        95
Merch       94
RainGear    86
Name: count, dtype: int64
[category cleanup] normalized case/punctuation, folded Apparel into Merch (6 -> 3 categories) (275 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [8]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert df['item'].isin(['cheeseburger', 'foam finger', 'rain poncho']).all()
assert df['category'].isin(['Food', 'Merch', 'RainGear']).all()
print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [9]:
df['revenue'] = df['qty'] * df['price']

by_category = (
    df.groupby('category')['revenue']
      .sum()
      .round(2)
      .sort_values(ascending=False)
)
print(by_category)
print()
print('total revenue:', round(df['revenue'].sum(), 2))

category
Food        1656.0
Merch       1572.0
RainGear    1512.0
Name: revenue, dtype: float64

total revenue: 4740.0


**What I would tell the vendor:** RainGear is the smallest at \$1,512.00, so it can be replenished more cautiously than the other two, and food makes the most money (\$1,656.00), so make sure it never sells out.

### TODO 8 — read back your log

In [10]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,duplicates,dropped exact duplicate rows,15
1,price to numeric,"stripped '$' and stray whitespace, converted t...",300
2,missing quantity,dropped rows with no quantity (unknown units s...,12
3,negative quantity,dropped refund rows (negative units) rather th...,13
4,category cleanup,"normalized case/punctuation, folded Apparel in...",275


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

**a)** The biggest change in revenue occurred the refund rows (negative quantity) were dropped. Prior to this step, revenue was \$4,594.50 (refunds were still deducting from the total), but after eliminating them, it increased to \$4,740.00, a change of \$145.50.

**b)** Instead of leaving it as a separate category eliminating it, I combined `Apparel` into `Merch`. If I hadn't folded it in, Merch would have shown \$856.50 and a new Apparel category would have shown \$715.50 instead of a total of \$1,572.00. A reasonable person would have kept `Apparel` distinct instead. In order to make the report comparable to Q2 of the pandas lab without creating a fourth bucket, I folded them together because the lab only requires three categories (Food, Merch, and RainGear) and `Apparel` doesn't fit any of the other two.